# Federated batch correction with FedRBE

FedRBE (`flimmaBatchCorrection` FeatureCloud app) is the federated equivalent of `limma::removeBatchEffect`. This notebook uses the FedRBE-style client folders prepared by `02_prepare_RBE_inputs.ipynb` for the four-client Quartet multiomics federation (`client_01_L01`, `client_02_L02`, `client_03_L05_L04`, `client_04_L03_L14`) and runs the federated correction.

Each client carries 12 or 24 libraries from one (or two) source labs; the `batch_col` inside each client is the original Quartet `batch`. The reference donor is `D6`. The reference batch is read from `before/fedrbe_client_groups.tsv`, written by step 02, so the federated and central corrections use the same prepared filtering and batch reference.


In [1]:
from pathlib import Path
import sys

WORKDIR = Path.cwd() if Path("figshare_data").exists() else Path("evaluation_data/multiomics")
REPO_ROOT = WORKDIR.resolve().parents[1]
if str(WORKDIR.resolve()) not in sys.path:
    sys.path.insert(0, str(WORKDIR.resolve()))

from fedrbe_multiomics_utils import MODALITIES

print(f"WORKDIR: {WORKDIR.resolve()}")
print(f"Modalities: {MODALITIES}")


WORKDIR: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/multiomics
Modalities: ['Transcriptomics', 'Proteomics', 'Metabolomics']


## FeatureCloud app run (main path)

The main correction path uses the `featurecloud.ai/bcorrect:latest` Docker image via the FeatureCloud Python API. Per-client folders (`intensities_log_UNION.tsv`, `design.tsv`, `config.yml`) are read from `before/<modality>/<client>/`; these are written by step 02 from the same filtered data as central correction. The corrected output is saved as `FedApp_corrected_data.tsv`. Requires the FeatureCloud Python package, Docker Python SDK, PyYAML, and a running Docker daemon with the local `featurecloud.ai/bcorrect:latest` image.


In [2]:
import pandas as pd
from copy import deepcopy

sys.path.insert(0, str(REPO_ROOT))
from evaluation_utils import featurecloud_api_extension as util
from evaluation_utils.featurecloud_api_extension import postprocess_results

# --- Step 1: use per-client input folders written by 02_prepare_RBE_inputs.ipynb ---
groups_path = WORKDIR / "before" / "fedrbe_client_groups.tsv"
if not groups_path.exists():
    raise FileNotFoundError(f"Missing {groups_path}. Run 02_prepare_RBE_inputs.ipynb first.")
client_summary = pd.read_csv(groups_path, sep="\t")
for client_path in client_summary["path"]:
    expected = WORKDIR / client_path / "intensities_log_UNION.tsv"
    if not expected.exists():
        raise FileNotFoundError(f"Missing prepared client matrix: {expected}")
print(f"Using prepared client folders from {groups_path}")
print(client_summary[["modality", "client", "position", "n_features", "all_na_feature_rows_dropped", "reference_batch"]])

# --- Step 2: run the FeatureCloud app for each modality ---
app_image_name = "featurecloud.ai/bcorrect:latest"
base_config = {
    "flimmaBatchCorrection": {
        "data_filename": "intensities_log_UNION.tsv",
        "design_filename": "design.tsv",
        "expression_file_flag": True,
        "index_col": "rowname",
        "batch_col": "batch",
        "covariates": ["D5", "F7", "M8"],
        "separator": "\t",
        "design_separator": "\t",
        "normalizationMethod": None,
        "smpc": False,
        "min_samples": 0,
        "position": None,
        "reference_batch": False,
    }
}


Using prepared client folders from /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/multiomics/before/fedrbe_client_groups.tsv
          modality             client  position  n_features  \
0  Transcriptomics      client_01_L01         0       26907   
1  Transcriptomics      client_02_L02         1       26907   
2  Transcriptomics  client_03_L05_L04         2       26907   
3       Proteomics      client_01_L01         0        2539   
4       Proteomics      client_02_L02         1        2539   
5       Proteomics  client_03_L05_L04         2        2539   
6     Metabolomics      client_01_L01         0          71   
7     Metabolomics      client_02_L02         1          71   
8     Metabolomics  client_03_L05_L04         2          71   

   all_na_feature_rows_dropped reference_batch  
0                            0           False  
1                            0           False  
2                            0     R_ILM_L5_B2  
3                            0       

In [3]:

for modality in MODALITIES:
    all_groups = pd.read_csv(groups_path, sep="\t")
    groups = all_groups[all_groups["modality"] == modality].reset_index(drop=True)
    clients = [str(WORKDIR / client_path) for client_path in groups["path"]]
    config_changes = []
    for _, row in groups.iterrows():
        reference = row["reference_batch"] if isinstance(row["reference_batch"], str) and row["reference_batch"] != "False" else False
        config_changes.append({
            "flimmaBatchCorrection": {
                "position": int(row["position"]),
                "reference_batch": reference,
                "smpc": False,
            }
        })
    experiment = util.Experiment(
        name=f"Multiomics {modality}",
        fc_data_dir=str(WORKDIR),
        clients=clients,
        app_image_name=app_image_name,
        config_files=[deepcopy(base_config) for _ in clients],
        config_file_changes=config_changes,
    )
    result_files_zipped, _, _ = experiment.run_test()
    result_path = WORKDIR / "after" / modality / "FedApp_corrected_data.tsv"
    result_path.parent.mkdir(parents=True, exist_ok=True)
    postprocess_results(experiment, result_files_zipped, str(result_path))
    print(f"{modality}: saved to {result_path}")


_______________EXPERIMENT_______________
Killing leftover containers...
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_featurecloudaibcorrectlatest_666647983', 'containerID': 'f20b49e31a65227f', 'coordinator': True, 'frontendUrl': '/app-traffic/048bc4ca54/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_featurecloudaibcorrectlatest_369938405', 'containerID': 'd22fe73f10dfcc76', 'coordinator': False, 'frontendUrl': '/app-traffic/e657049abd/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 2, 'name': 'fc_featurecloudaibcorrectlatest_203474353', 'containerID': '5bcdc198fcc49ed0', 'coordinator': False, 'frontendUrl': '/app-traffic/9f309ed663/', 'message': 'terminal', 'state': None, 'progress': None}]
TEST DONE:
_______________EXPERIMENT FINISHED SUCCESSFULLY_______________
Saved concatenated result ((26907, 48)) to /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_data/multiomics/after/Transcriptomics/

## Output files

Per-modality FeatureCloud app outputs are saved as `FedApp_corrected_data.tsv` under `after/<modality>/`.


In [4]:
# All paths should exist after a successful run.
for modality in MODALITIES:
    out_dir = WORKDIR / "after" / modality
    app_path = out_dir / "FedApp_corrected_data.tsv"
    print(f"{modality}: FedApp_corrected_data.tsv={app_path.exists()}")


Transcriptomics: FedApp_corrected_data.tsv=True
Proteomics: FedApp_corrected_data.tsv=True
Metabolomics: FedApp_corrected_data.tsv=True


## Optional local FedRBE-equivalent simulation

Set `RUN_FEDSIM = True` to run an in-process simulation of the federated correction without Docker. `run_all_fedsim` uses the client folders prepared by step 02, performs the same XTX/XTY aggregation and batch-effect subtraction as the FeatureCloud app locally, and writes results as `FedSim_corrected_data.tsv` per modality. The joint k-means matrix is built later by the R notebook `00_build_kmeans_matrices.ipynb` in the clustering folder (it reads `FedApp_corrected_data.tsv` if present, otherwise falls back to `FedSim_corrected_data.tsv`).


In [ ]:
RUN_FEDSIM = False

if RUN_FEDSIM:
    from fedrbe_multiomics_utils import run_all_fedsim
    correction_summary = run_all_fedsim(WORKDIR)
    print(correction_summary)

    for modality in MODALITIES:
        out_dir = WORKDIR / "after" / modality
        fed_path = out_dir / "FedSim_corrected_data.tsv"
        print(f"{modality}: FedSim_corrected_data.tsv={fed_path.exists()}")

    print(f"Summary: {(WORKDIR / 'after' / 'fedsim_correction_summary.tsv').exists()}")
else:
    print("FedSim run skipped. Set RUN_FEDSIM = True to run the local in-process simulation.")
